# 🛡️ Drishti Kavach: Universal 'RailDrishti' Training Notebook

This notebook is **Universal & Hybrid-Smart**. It runs seamlessly on:
1. **Google Colab Cloud GPU** (NVIDIA Tesla T4 / A100)
2. **Local Runtime connected from Laptop** (Apple Silicon MPS / Windows / Linux CPU/GPU)

It auto-detects your environment, hardware accelerator, and dataset paths with live **Color-Coded Real-World Accuracy Monitoring**.

In [ ]:
# 1. Hardware & Environment Check
import torch, os, platform

is_colab_cloud = os.path.exists('/content') and not os.path.exists('dataset_rail-drishti')
print("=" * 60)
print(" DRISHTI KAVACH: HARDWARE & ENVIRONMENT AUDIT")
print("=" * 60)
print(f" • System OS      : {platform.system()} ({platform.release()})")
print(f" • PyTorch Version: {torch.__version__}")

if torch.cuda.is_available():
    device = 0
    print(f" • Compute Device : NVIDIA CUDA GPU ({torch.cuda.get_device_name(0)})")
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    device = 'mps'
    print(" • Compute Device : Apple Silicon GPU (Metal Performance Shaders - MPS)")
else:
    device = 'cpu'
    print(" • Compute Device : Multi-core CPU")

print(f" • Runtime Mode   : {'Google Colab Cloud' if is_colab_cloud else 'Local Laptop Runtime'}")
print("=" * 60)

In [ ]:
# 2. Install Ultralytics and dependencies
!pip install ultralytics -q

In [ ]:
# 3. Dataset Setup & Path Resolution (Auto-handles Cloud vs Local)
import os, zipfile, yaml

# Check if running on Local Laptop Runtime
if os.path.exists('dataset_rail-drishti'):
    dataset_path = os.path.abspath('dataset_rail-drishti')
    config_path = os.path.abspath('configs/raildrishti_dataset.yaml')
    print(f"[+] Local Runtime Detected! Using existing dataset directly at:\n    {dataset_path}")

# Check if running on Cloud Colab
else:
    dataset_path = '/content/dataset_rail-drishti'
    config_path = '/content/raildrishti_colab.yaml'
    local_zip = '/content/raildrishti_colab.zip'
    drive_zip = '/content/drive/MyDrive/raildrishti_colab.zip'

    if os.path.exists(local_zip):
        print("[+] Found zip in session storage! Unzipping...")
        with zipfile.ZipFile(local_zip, 'r') as zip_ref:
            zip_ref.extractall('/content/')
    else:
        try:
            from google.colab import drive
            drive.mount('/content/drive')
            if os.path.exists(drive_zip):
                print("[+] Found zip in Google Drive! Unzipping...")
                with zipfile.ZipFile(drive_zip, 'r') as zip_ref:
                    zip_ref.extractall('/content/')
            else:
                print("[!] Please upload 'raildrishti_colab.zip' to Google Drive (MyDrive)!")
        except Exception as e:
            print("[!] Drive Mount error:", e)

    # Generate Colab Cloud YAML
    data_dict = {
        'path': dataset_path,
        'train': 'images/train',
        'val': 'images/val',
        'names': {
            0: 'Rail_Track_Bed', 1: 'Rail_Lines', 2: 'Branch', 3: 'IronRod',
            4: 'Barrel', 5: 'Boulder', 6: 'Jerrycan', 7: 'Person',
            8: 'Cattle', 9: 'Animal', 10: 'Vehicle'
        }
    }
    with open(config_path, 'w') as f:
        yaml.dump(data_dict, f, sort_keys=False)
    print(f"[+] Created config at {config_path}")

In [ ]:
# 4. Train Unified 'RailDrishti' Model (YOLO11-seg with Minimal Color-Coded Monitor)
import warnings, logging, os, torch
warnings.filterwarnings('ignore')
os.environ['PYTHONWARNINGS'] = 'ignore'
logging.getLogger('ultralytics').setLevel(logging.WARNING)

from ultralytics import YOLO

class EpochStatusMonitor:
    TARGET_BOX_MAP = 85.0
    TARGET_SEG_MAP = 90.0
    TARGET_OVERALL_MAP = 85.0

    def __init__(self):
        self.best_map = 0.0
        self.best_epoch = 0
        self.prev_loss = None
        self.prev_map = None

    def on_fit_epoch_end(self, trainer):
        epoch = trainer.epoch + 1
        total_epochs = trainer.epochs
        metrics = getattr(trainer, 'metrics', {}) or {}

        box_map50 = (metrics.get('metrics/mAP50(B)', 0.0) or 0.0) * 100.0
        seg_map50 = (metrics.get('metrics/mAP50(M)', 0.0) or 0.0) * 100.0
        overall_map50 = (box_map50 + seg_map50) / 2.0 if (box_map50 > 0 and seg_map50 > 0) else (box_map50 or seg_map50 or 0.0)

        loss_val = None
        if hasattr(trainer, 'tloss') and trainer.tloss is not None:
            try:
                loss_val = float(trainer.tloss.mean()) if hasattr(trainer.tloss, 'mean') else float(trainer.tloss)
            except Exception:
                loss_val = None

        is_new_best = False
        if overall_map50 > self.best_map and overall_map50 > 2.0:
            self.best_map = overall_map50
            self.best_epoch = epoch
            is_new_best = True

        # Color gradient: Red (Worst) -> Green (Best)
        if overall_map50 >= 90.0:
            state_tag = "🏆 [BEST / DEPLOYMENT READY]"
            color = "\033[1;92m"
        elif overall_map50 >= 80.0:
            state_tag = "🌟 [GOOD / OPERATIONAL GRADE]"
            color = "\033[92m"
        elif overall_map50 >= 65.0:
            state_tag = "📈 [APPROACHING TARGET]"
            color = "\033[93m"
        elif overall_map50 >= 45.0:
            state_tag = "🔄 [LEARNING & IMPROVING]"
            color = "\033[95m"
        else:
            state_tag = "🌱 [UNDER-TRAINED / INITIALIZING]"
            color = "\033[91m"

        box_diff = box_map50 - self.TARGET_BOX_MAP
        seg_diff = seg_map50 - self.TARGET_SEG_MAP
        overall_diff = overall_map50 - self.TARGET_OVERALL_MAP

        box_tag = f"[{'+' if box_diff >= 0 else ''}{box_diff:4.1f}%]"
        seg_tag = f"[{'+' if seg_diff >= 0 else ''}{seg_diff:4.1f}%]"
        overall_tag = f"[{'+' if overall_diff >= 0 else ''}{overall_diff:4.1f}%]"

        loss_str = "N/A"
        if loss_val is not None:
            diff_str = ""
            if self.prev_loss is not None:
                d = loss_val - self.prev_loss
                diff_str = f" ({'+' if d > 0 else ''}{d:.3f})"
            loss_str = f"{loss_val:.4f}{diff_str}"

        peak_str = f"{self.best_map:4.1f}% (Ep {self.best_epoch})" + (" *" if is_new_best else "")
        reset = "\033[0m"

        print(f"\n{color}┌─── EPOCH [{epoch:02d}/{total_epochs:02d}] MODEL STATE: {state_tag} ─────────────────────────────┐{reset}")
        print(f"{color}│{reset}  • OVERALL ACCURACY : {color}{overall_map50:5.1f}%{reset} (Target: {self.TARGET_OVERALL_MAP:.1f}% {overall_tag}) | Peak: {peak_str}")
        print(f"{color}│{reset}  • Obstacle Box mAP : {box_map50:5.1f}% / {self.TARGET_BOX_MAP:.1f}% Target {box_tag:<8} | Loss: {loss_str}")
        print(f"{color}│{reset}  • Track Mask mAP   : {seg_map50:5.1f}% / {self.TARGET_SEG_MAP:.1f}% Target {seg_tag:<8} | Status: NOMINAL")
        print(f"{color}└───────────────────────────────────────────────────────────────────────────────┘{reset}\n")

        self.prev_loss = loss_val
        self.prev_map = overall_map50

# Auto-detect compute device (0 for CUDA, 'mps' for Apple Silicon, 'cpu' for fallback)
if torch.cuda.is_available():
    train_device = 0
    img_size = 1024
    batch_size = 8
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    train_device = 'mps'
    img_size = 640
    batch_size = 8
else:
    train_device = 'cpu'
    img_size = 640
    batch_size = 4

print(f"[+] Launching training on device: {train_device} (imgsz={img_size}, batch={batch_size})")

model = YOLO('yolo11s-seg.pt')
monitor = EpochStatusMonitor()
model.add_callback('on_fit_epoch_end', monitor.on_fit_epoch_end)

results = model.train(
    data=config_path,
    epochs=40,
    imgsz=img_size,
    batch=batch_size,
    device=train_device,
    name='RailDrishti_Training',
    workers=2,
    optimizer='AdamW',
    lr0=0.001,
    lrf=0.01,
    cache=False,
    amp=True,
    save=True,
    patience=20,
    box=7.5,
    cls=1.2,
    dfl=1.8,
    verbose=True
)

In [ ]:
# 5. Save Model to models/ Folder & Google Drive
import os, shutil, glob

found_bests = glob.glob('runs/**/weights/best.pt', recursive=True)
if found_bests:
    found_bests.sort(key=os.path.getmtime, reverse=True)
    best_pt = found_bests[0]
    
    # 1. Save to models/ folder
    os.makedirs('models', exist_ok=True)
    shutil.copy(best_pt, 'models/RailDrishti.pt')
    print(f"[+] Model weights saved locally to: {os.path.abspath('models/RailDrishti.pt')}")
    
    # 2. If Google Drive is mounted, backup there as well
    if os.path.exists('/content/drive/MyDrive'):
        shutil.copy(best_pt, '/content/drive/MyDrive/RailDrishti.pt')
        print("[+] Backup copy saved to Google Drive: MyDrive/RailDrishti.pt")
    
    # 3. If in Colab Cloud, trigger browser download
    if os.path.exists('/content') and not os.path.exists('src'):
        try:
            from google.colab import files
            files.download('models/RailDrishti.pt')
        except Exception:
            pass
else:
    print("[!] Training output weights not found in runs/.")